In [ ]:
using Plots
using Plots.PlotMeasures
using StatsPlots, KernelDensity
using Statistics

include("../src/fxio.jl")
include("../src/kabsch_umeyama.jl")


In [ ]:
k3mers[1]


In [ ]:
pdb_bb

In [ ]:
scoppath = "../../scop40NR/"  #
virtual_torsions = reshape(Float64[], 0, 3)
sscolors = []
k4mers = []
k3mers = []
k3mers_caonly = []
k3angles = []
ca_ca = []

for pdbfile in readdir(scoppath)

    try
        pdbpath = joinpath(scoppath, pdbfile)                
        pdb = readpdb_calpha(pdbpath)       
        pdb_bb = readpdb_backbone(pdbpath)        
        xyz = pdb2xyz(pdb)
        xyz_bb = pdb2xyz(pdb_bb)
        kmers = coords2kmers(xyz, 4, "ca")
        k3mers_bb = coords2kmers(xyz_bb, 3, "bb")
        k3mers_ca = coords2kmers(xyz, 3, "ca")

        vtors = zeros(Float64, length(kmers), 3)
        for (i, kmer) in enumerate(kmers)
            vtors[i,1:3] = vtor(kmer)
        end

        for (i, kmer) in enumerate(k3mers_ca)
            push!(k3mers_caonly, kmer)
            push!(k3angles, k3angle(kmer))
        end

        for (i, kmer) in enumerate(k3mers_bb)
            push!(k3mers, kmer)
        end

        if !missing_residues(pdb)
            if (size(vtors, 1) == size(kmers, 1)) 
                if !any(isnan.(vtors))
                    virtual_torsions = vcat(virtual_torsions, vtors)
                    k4mers = vcat(k4mers, kmers)

                    for i in 1:size(xyz, 1)-1
                        p1 = xyz[i,:]
                        p2 = xyz[i+1,:]
                        push!(ca_ca, euclid_dist(p1, p2))
                    end
                end        
            end
        end
    
    catch e
        println(e)
        println(pdbfile)
    
    end
end

println(size(virtual_torsions, 1))

In [ ]:
scoppath = "../../scop40NR/"  #
k3mers = []
k3mers_caonly = []
k3angles = []

for pdbfile in readdir(scoppath)

    try
        pdbpath = joinpath(scoppath, pdbfile)                
        pdb = readpdb_calpha(pdbpath)       
        pdb_bb = readpdb_backbone(pdbpath)        
        xyz = pdb2xyz(pdb)
        xyz_bb = pdb2xyz(pdb_bb)
        k3mers_bb = coords2kmers(xyz_bb, 3, "bb")
        k3mers_ca = coords2kmers(xyz, 3, "ca")


        if !missing_residues(pdb)     
            for (i, kmer) in enumerate(k3mers_ca)
                push!(k3mers_caonly, kmer)
                angles = k3angle(kmer)
                push!(k3angles, angles)
            end

            for (i, kmer) in enumerate(k3mers_bb)
                push!(k3mers, kmer)
            end
        end
    
    catch e
        println(e)
        println(pdbfile)
    
    end
end



In [ ]:
l = 100000 #534380
s = 500

msize = Int(l/s)

k3_mers = [k for k in k3mers][1:s:l]
angles = k3angles[1:s:l,:]


k3_pairs_rmsd = []
k3_delta = [] 

# function angle_delta(a, b)
#     delta = a - b
#     return abs.(mod.(delta .+ 180, 360) .- 180)
# end

function  angle_delta(a, b)
    return abs.(a - b)
    
end


for i in 1:msize
    for j in i+1:msize
        try
            rms = rmsd(k3_mers[i], k3_mers[j])
            delta_angle = angle_delta(angles[i], angles[j])[1]
            push!(k3_delta, delta_angle)     
            push!(k3_pairs_rmsd, rms)     
                   
        catch err
            println(err)
        end
    end
end



In [ ]:
function scatter_corr(x, y)    
    s = scatter(x, y, label="R^2 = $(round(cor(x, y), sigdigits=2))", markershape=:circle, markersize = 0.9, alpha=0.5)
    return s
end
h1 = histogram(k3_delta, bins=100, label="Delta angles", legend=:topleft, title="Histogram of delta angles", xlabel="Delta angles", ylabel="Counts", size=(400, 400), dpi=100, margins = 6mm, ytickfontsize=6, xtickfontsize=6)
h2 = histogram(k3_pairs_rmsd, bins=100, label="RMSD", legend=:topleft, title="Histogram of RMSD", xlabel="RMSD", ylabel="Counts", size=(400, 400), dpi=100, margins = 6mm, ytickfontsize=6, xtickfontsize=6)
s =scatter_corr(k3_pairs_rmsd, k3_delta)


plot(h1, h2, s, layout=(1,3), size=(1000, 300), dpi=100, margins = 6mm, ytickfontsize=6, xtickfontsize=6, 
    ylabel="RMSD", 
    xlabel="Delta angles")

In [ ]:
l = 100000 #534380
s = 100

msize = Int(l/s)

k4_mers = [k for k in k4mers][1:s:l]
v_tors = virtual_torsions[1:s:l,:]
#v_tors[:,1] = v_tors[:,1].+180

k4_vs_k4 = zeros(msize, msize)
rmsd_pairs = []
vtor_pairs = []
vtor_pairs_rmsd = []
vtor_delta = [] 

function angle_delta(a, b)
    delta = a - b
    return abs.(mod.(delta .+ 180, 360) .- 180)
end

for i in 1:msize
    for j in i+1:msize
        try
            rms = rmsd(k4_mers[i], k4_mers[j])
            k4_vs_k4[i,j] = rms
            # push!(rmsd_pairs, ([i,j], rms))
            # push!(dihs_delta, ([i,j], abs(v_tors[i]-v_tors[j])))
            delta_vrot = angle_delta(v_tors[i,:], v_tors[j,:])
            push!(rmsd_pairs, rms)
            push!(vtor_delta, delta_vrot)
            push!(vtor_pairs, ([i,j], (v_tors[i,:], v_tors[j,:])))
            push!(vtor_pairs_rmsd, vcat(v_tors[i,:], v_tors[j,:],rms))
            
        catch err
            println(err)
        end
    end
end


vtor_delta = hcat(vtor_delta...)'
vtor_pairs_rmsd = hcat(vtor_pairs_rmsd...)'
deltas = hcat(vtor_delta, rmsd_pairs)

x1 = deltas[:,1]
x2 = deltas[:,2]
x3 = deltas[:,3]
y  = deltas[:,4]

size(deltas)

In [ ]:
CSV.write("../data/deltas_vs_rmsd.csv", DataFrame([x1, x2, x3, y], :auto))
CSV.write("../data/vtors_vs_rmsd.csv", DataFrame(vtor_pairs_rmsd, :auto))

In [ ]:
s1 = scatter_corr(x1, y)
s2 = scatter_corr(x2, y)
s3 = scatter_corr(x3, y)
s4 = scatter_corr(x2, x3)

plot(s1, s2, s3, s4, layout=(1, 4), size=(1200, 300), dpi=100, margins = 6mm, ytickfontsize=6, xtickfontsize=6, 
    ylabel=["RMSD" "RMSD" "RMSD" "Delta Angles 1"], 
    xlabel=["Delta dihidrals" "Delta Angles 1" "Delta Angles 2" "Delta Angles 2"])

In [ ]:
pred = predx1
# Compute RMSE
rmse = sqrt(mean((pred - y).^2))
pirsonr = cor(pred, y)
println("RMSE: ", rmse)
println("R^2: ", pirsonr)

# Alternatively, compute R²

scatter_corr(predx9[1:1:end], y[1:1:end])